# 유저별 이벤트 로그 SQL 생성 (first_save_history용)

## 대상 이벤트
- coupon_received, coupon_used, ad_click, ad_exposure, purchase_button_click

## 유저
- user0001~user0100, user_id = 숫자+14 (user0001=15, user0100=114)
- 비로그인 없음

## 날짜 범위
- 2026-05-17 ~ 2026-06-16

## 유저별 이벤트 수 제약
- ad_click: 1~5 랜덤
- ad_exposure: ad_click보다 많게 (ad_click + 1~3)
- purchase_button_click: ad_click 이하 랜덤
- coupon_received: 1~5 랜덤 (유저당 같은 쿠폰 중복 수령 불가)
- coupon_used: coupon_received에서 발급된 쿠폰 중 일부 (만료일 이내)

## 쿠폰
- 8종: WELCOME5000, COMEBACK20, REGULAR5000, FIRSTORDER10,
       SPRING20(3-5월), SUMMER10000(6-8월), AUTUMN20(9-11월), WINTER10000(12-2월)
- RATE 타입은 discountAmount=0 (FE 버그 재현), FIXED 타입은 실제 금액

## adId
- 1~100 중 3, 5, 6 제외

## 사전 조건
- products.csv가 같은 디렉토리에 있어야 함

In [1]:
import random
import json
import uuid
import pandas as pd
from datetime import datetime, timedelta

In [2]:
# ───────────────────────────────────────────
# 설정값
# ───────────────────────────────────────────
START_DATE = datetime(2026, 5, 17, 0, 0, 0)
END_DATE   = datetime(2026, 6, 16, 23, 59, 59)

USER_COUNT = 100  # user0001 ~ user0100
USER_ID_OFFSET = 14  # user0001 -> user_id=15 (+14)

# 유저별 이벤트 수 범위
AD_CLICK_MIN, AD_CLICK_MAX         = 1, 5
AD_EXPOSURE_EXTRA_MIN              = 1  # ad_click + 이 값 이상
AD_EXPOSURE_EXTRA_MAX              = 3
COUPON_RECEIVED_MIN                = 1
COUPON_RECEIVED_MAX                = 5

VALID_CATEGORIES = ['디지털/가전', '패션의류', '패션잡화', '화장품/미용', '식품', '생활/건강', '스포츠/레저', '가구/인테리어']

AD_IDS = [i for i in range(1, 101) if i not in (3, 5, 6)]

COUPONS = [
    {'code': 'WELCOME5000',  'discount_amount': 5000,  'discount_type': 'FIXED', 'expired_days': 30, 'season': None},
    {'code': 'COMEBACK20',   'discount_amount': 20,    'discount_type': 'RATE',  'expired_days': 7,  'season': None},
    {'code': 'REGULAR5000',  'discount_amount': 5000,  'discount_type': 'FIXED', 'expired_days': 7,  'season': None},
    {'code': 'FIRSTORDER10', 'discount_amount': 10,    'discount_type': 'RATE',  'expired_days': 7,  'season': None},
    {'code': 'SPRING20',     'discount_amount': 20,    'discount_type': 'RATE',  'expired_days': 7,  'season': [3, 4, 5]},
    {'code': 'SUMMER10000',  'discount_amount': 10000, 'discount_type': 'FIXED', 'expired_days': 7,  'season': [6, 7, 8]},
    {'code': 'AUTUMN20',     'discount_amount': 20,    'discount_type': 'RATE',  'expired_days': 7,  'season': [9, 10, 11]},
    {'code': 'WINTER10000',  'discount_amount': 10000, 'discount_type': 'FIXED', 'expired_days': 7,  'season': [12, 1, 2]},
]

In [3]:
# products.csv 로드
products_df = pd.read_csv('products.csv')
products_df = products_df[
    (products_df['is_active'] == 1) &
    (products_df['product_category'].isin(VALID_CATEGORIES))
][['product_id', 'name', 'product_category', 'min_price', 'max_price']].dropna(subset=['product_id', 'name'])
product_pool = products_df.to_dict('records')
print(f'✅ products.csv 로드 완료 → {len(product_pool)}개')

✅ products.csv 로드 완료 → 2476개


In [4]:
def random_datetime(start, end):
    delta = end - start
    return start + timedelta(seconds=random.randint(0, int(delta.total_seconds())))

def random_datetime_in_months(start, end, months):
    candidates = []
    for ym in range(start.year * 12 + start.month - 1, end.year * 12 + end.month):
        y, m = divmod(ym, 12)
        m += 1
        if m in months:
            candidates.append((y, m))
    if not candidates:
        return random_datetime(start, end)
    while True:
        y, m = random.choice(candidates)
        day = random.randint(1, 28)
        dt  = datetime(y, m, day, random.randint(0,23), random.randint(0,59), random.randint(0,59))
        if start <= dt <= end:
            return dt

def format_kst(dt):
    return dt.strftime('%Y-%m-%dT%H:%M:%S.') + f"{dt.microsecond // 1000:03d}+09:00"

def format_history_ts(dt):
    return dt.strftime('%Y-%m-%d %H:%M:%S.%f')

def format_date(dt):
    return dt.strftime('%Y-%m-%d')

In [5]:
all_rows = []          # (history_ts, json_log) 전체
received_map = []      # coupon_used 생성용

for user_num in range(1, USER_COUNT + 1):
    user_login_id = f'user{user_num:04d}'
    user_id       = user_num + USER_ID_OFFSET

    # ── 이벤트 수 결정 ──
    ad_click_count    = random.randint(AD_CLICK_MIN, AD_CLICK_MAX)
    ad_exposure_count = ad_click_count + random.randint(AD_EXPOSURE_EXTRA_MIN, AD_EXPOSURE_EXTRA_MAX)
    purchase_count    = random.randint(1, ad_click_count)
    received_count    = random.randint(COUPON_RECEIVED_MIN, COUPON_RECEIVED_MAX)

    # ── ad_click ──
    for _ in range(ad_click_count):
        event_ts   = random_datetime(START_DATE, END_DATE)
        history_ts = event_ts + timedelta(seconds=1)
        product    = random.choice(product_pool)
        json_log   = json.dumps({
            'event_name':      'ad_click',
            'adId':            random.choice(AD_IDS),
            'productName':     product['name'],
            'productId':       str(product['product_id']),
            'productCategory': product['product_category'],
            'user_id':         user_id,
            'user_login_id':   user_login_id,
            'client_uuid':     str(uuid.uuid4()),
            'event_timestamp': format_kst(event_ts)
        }, ensure_ascii=False)
        all_rows.append((history_ts, json_log))

    # ── ad_exposure ──
    for _ in range(ad_exposure_count):
        event_ts   = random_datetime(START_DATE, END_DATE)
        history_ts = event_ts + timedelta(seconds=1)
        product    = random.choice(product_pool)
        json_log   = json.dumps({
            'event_name':      'ad_exposure',
            'adId':            random.choice(AD_IDS),
            'productName':     product['name'],
            'productId':       str(product['product_id']),
            'productCategory': product['product_category'],
            'user_id':         user_id,
            'user_login_id':   user_login_id,
            'client_uuid':     str(uuid.uuid4()),
            'event_timestamp': format_kst(event_ts)
        }, ensure_ascii=False)
        all_rows.append((history_ts, json_log))

    # ── purchase_button_click ──
    for _ in range(purchase_count):
        event_ts   = random_datetime(START_DATE, END_DATE)
        history_ts = event_ts + timedelta(seconds=1)
        product    = random.choice(product_pool)
        min_p = product.get('min_price') or 0
        max_p = product.get('max_price') or min_p
        if min_p > max_p:
            min_p, max_p = max_p, min_p
        approved_amount = random.randint(int(min_p), int(max_p)) if min_p != max_p else int(min_p)
        json_log = json.dumps({
            'event_name':      'purchase_button_click',
            'productName':     product['name'],
            'productId':       str(product['product_id']),
            'productCategory': product['product_category'],
            'approvedAmount':  approved_amount,
            'user_id':         user_id,
            'user_login_id':   user_login_id,
            'client_uuid':     str(uuid.uuid4()),
            'event_timestamp': format_kst(event_ts)
        }, ensure_ascii=False)
        all_rows.append((history_ts, json_log))

    # ── coupon_received ──
    # 유저당 중복 쿠폰 없게 최대 received_count 만큼 샘플링
    available_coupons = COUPONS.copy()
    # 시즌 쿠폰 중 현재 날짜 범위에 해당하는 것만 포함
    valid_coupons = []
    for c in available_coupons:
        if c['season'] is None:
            valid_coupons.append(c)
        else:
            # START_DATE~END_DATE 범위 내에 해당 시즌 월이 있는지 확인
            has_season_month = any(
                m in c['season']
                for m in range(1, 13)
                if any(
                    datetime(y, m, 1) >= START_DATE.replace(day=1) and
                    datetime(y, m, 28) <= END_DATE
                    for y in [START_DATE.year, END_DATE.year]
                )
            )
            if has_season_month:
                valid_coupons.append(c)

    selected_coupons = random.sample(valid_coupons, min(received_count, len(valid_coupons)))

    user_received = []
    for coupon in selected_coupons:
        if coupon['season']:
            event_ts = random_datetime_in_months(START_DATE, END_DATE, coupon['season'])
        else:
            event_ts = random_datetime(START_DATE, END_DATE)
        history_ts      = event_ts + timedelta(seconds=1)
        discount_amount = 0 if coupon['discount_type'] == 'RATE' else coupon['discount_amount']
        expiry_date     = format_date(event_ts + timedelta(days=coupon['expired_days']))
        json_log = json.dumps({
            'event_name':      'coupon_received',
            'couponCode':      coupon['code'],
            'discountAmount':  discount_amount,
            'expiryDate':      expiry_date,
            'user_id':         user_id,
            'user_login_id':   user_login_id,
            'client_uuid':     str(uuid.uuid4()),
            'event_timestamp': format_kst(event_ts)
        }, ensure_ascii=False)
        all_rows.append((history_ts, json_log))
        user_received.append({
            'user_id':           user_id,
            'user_login_id':     user_login_id,
            'coupon_code':       coupon['code'],
            'discount_amount':   discount_amount,
            'expired_days':      coupon['expired_days'],
            'received_event_ts': format_kst(event_ts),
        })

    # ── coupon_used: received보다 적게 ──
    if len(user_received) > 1:
        used_count = random.randint(1, len(user_received) - 1)
        used_records = random.sample(user_received, used_count)
        for record in used_records:
            received_ts = datetime.strptime(record['received_event_ts'][:-6], '%Y-%m-%dT%H:%M:%S.%f')
            earliest    = received_ts + timedelta(seconds=60)
            latest      = received_ts + timedelta(days=record['expired_days'])
            if latest > END_DATE:
                latest = END_DATE
            if earliest >= latest:
                continue
            delta_sec = int((latest - earliest).total_seconds())
            event_ts  = earliest + timedelta(seconds=random.randint(0, delta_sec))
            history_ts = event_ts + timedelta(seconds=1)
            json_log = json.dumps({
                'event_name':      'coupon_used',
                'couponCode':      record['coupon_code'],
                'discountAmount':  record['discount_amount'],
                'user_id':         record['user_id'],
                'user_login_id':   record['user_login_id'],
                'client_uuid':     str(uuid.uuid4()),
                'event_timestamp': format_kst(event_ts)
            }, ensure_ascii=False)
            all_rows.append((history_ts, json_log))

print(f'✅ 총 {len(all_rows)}개 이벤트 로그 생성 완료')

✅ 총 1417개 이벤트 로그 생성 완료


In [6]:
# SQL 생성 및 저장
lines  = ['INSERT INTO first_save_history (history_timestamp, json_log) VALUES']
values = []

for history_ts, json_log in all_rows:
    ts_str  = format_history_ts(history_ts)
    escaped = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('user_event_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {len(all_rows)}개 로그 SQL 생성 완료 → user_event_logs.sql')

✅ 1417개 로그 SQL 생성 완료 → user_event_logs.sql


In [7]:
# ── 미리보기 ──
print('=== USER EVENT SQL (앞 500자) ===')
print(sql[:500])

=== USER EVENT SQL (앞 500자) ===
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2026-06-04 15:14:44.000000', '{"event_name": "ad_click", "adId": 43, "productName": "글로벌이노스 나이팅핏 TPE 요가매트", "productId": "1677", "productCategory": "스포츠/레저", "user_id": 15, "user_login_id": "user0001", "client_uuid": "07ce894d-a2e2-43a6-be8f-7e2d1ab5c306", "event_timestamp": "2026-06-04T15:14:43.000+09:00"}'),
  ('2026-06-16 17:00:49.000000', '{"event_name": "ad_click", "adId": 78, "productName": "나이키 에어맥스 키높이 굽높은 어글리 여성 운동화
